# Advanced Topics

This notebook covers advanced features: cross-analysis, statistical comparisons, and complex workflows.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pyMyriad import AnalysisTree, simple_table
from pyMyriad.tabular import format_statistics, flatten, tabulate

In [ ]:
# Create sample data
np.random.seed(42)
df = pd.DataFrame({
    "Gender": np.random.choice(["M", "F"], 200),
    "Country": np.random.choice(["US", "UK", "Canada"], 200),
    "Age": np.random.randint(18, 70, 200),
    "Income": np.random.normal(50000, 15000, 200),
})

## Cross-Analysis with Reference Group

Use `cross_analyze_by()` to compare each group against a reference group.

In [ ]:
# Compare each country against US as reference
atree = AnalysisTree()\
    .split_by("df.Gender")\
    .split_by("df.Country")\
    .cross_analyze_by(
        ref_lvl="US",
        mean_current=lambda df, ref_df: np.mean(df.Income),
        mean_reference=lambda df, ref_df: np.mean(ref_df.Income),
        difference=lambda df, ref_df: np.mean(df.Income) - np.mean(ref_df.Income),
        n_current=lambda df, ref_df: len(df),
        n_reference=lambda df, ref_df: len(ref_df)
    )

result = atree.run(df)
simple_table(result)

## Cross-Analysis with Statistical Tests

Compute p-values comparing groups using t-tests.

In [ ]:
# Function to perform t-test and return p-value
def ttest_pvalue(df, ref_df):
    if len(df) < 2 or len(ref_df) < 2:
        return np.nan
    t_stat, p_val = stats.ttest_ind(df.Income, ref_df.Income)
    return p_val

# Cross-analysis with p-values
atree2 = AnalysisTree()\
    .split_by("df.Gender")\
    .split_by("df.Country")\
    .cross_analyze_by(
        ref_lvl="US",
        mean_diff=lambda df, ref_df: np.mean(df.Income) - np.mean(ref_df.Income),
        p_value=ttest_pvalue,
        n_current=lambda df, ref_df: len(df),
        n_ref=lambda df, ref_df: len(ref_df)
    )

result2 = atree2.run(df)
simple_table(result2)

## Format Statistical Comparisons

Create publication-ready formatted comparison statistics with p-values.

In [ ]:
# Format the p-value with significance stars
def format_pvalue(p):
    if np.isnan(p):
        return "-"
    elif p < 0.001:
        return f"{p:.4f}***"
    elif p < 0.01:
        return f"{p:.3f}**"
    elif p < 0.05:
        return f"{p:.3f}*"
    else:
        return f"{p:.3f}"

# Format mean difference with p-value
format_statistics(
    result2,
    format_spec="{mean_diff:.0f}",
    stat_name="Difference",
    inplace=True
)

# Get the flattened data to manually format p-values
flat_df = flatten(result2)
flat_df['p_formatted'] = flat_df['p_value'].apply(format_pvalue)

# Display selected columns
flat_df[['path', 'label', 'Difference', 'p_formatted', 'n_current', 'n_ref']].head(10)

## Complex Multi-Level Analysis with Tabulate

Create wide-format pivot tables using `tabulate()`.

In [ ]:
# Create multi-level analysis
atree3 = AnalysisTree()\
    .split_by("df.Gender")\
    .split_by("df.Country")\
    .analyze_by(
        mean=lambda df: np.mean(df.Income),
        sd=lambda df: np.std(df.Income),
        n=lambda df: len(df)
    )

result3 = atree3.run(df)

# Format statistics first
format_statistics(result3, format_spec="{mean:.0f} ± {sd:.0f}", stat_name="mean_sd", inplace=True)

# Create wide-format table with tabulate
wide_table = tabulate(result3, by="df.Country", statistics=["mean_sd", "n"])
wide_table

## Combining Flatten and Custom Analysis

Use `flatten()` to get data in long format for custom processing.

In [ ]:
# Flatten results for custom analysis
flat_df = flatten(result3)

# Filter and process
print("Columns available:")
print(flat_df.columns.tolist())
print("\nFirst few rows with key columns:")
flat_df[['path', 'label', 'mean', 'sd', 'n']].head()

## Advanced: Nested Cross-Analysis

Perform cross-analysis at multiple levels.

In [ ]:
# Create age groups and perform nested cross-analysis
atree4 = AnalysisTree()\
    .split_by(
        label="Age Group",
        young=lambda df: df.Age < 40,
        older=lambda df: df.Age >= 40
    )\
    .split_by("df.Gender")\
    .cross_analyze_by(
        ref_lvl="M",
        mean_female=lambda df, ref_df: np.mean(df.Income) if len(df) > 0 else np.nan,
        mean_male=lambda df, ref_df: np.mean(ref_df.Income) if len(ref_df) > 0 else np.nan,
        gender_diff=lambda df, ref_df: np.mean(df.Income) - np.mean(ref_df.Income) if len(df) > 0 and len(ref_df) > 0 else np.nan
    )

result4 = atree4.run(df)
simple_table(result4)

## Complete Workflow: Analysis to Publication Table

A complete example from data to publication-ready output.

In [ ]:
# 1. Define and run analysis
atree_final = AnalysisTree()\
    .split_by("df.Gender")\
    .split_by("df.Country")\
    .analyze_by(
        mean=lambda df: np.mean(df.Income),
        sd=lambda df: np.std(df.Income),
        n=lambda df: len(df),
        ci_lower=lambda df: np.mean(df.Income) - 1.96 * np.std(df.Income) / np.sqrt(len(df)),
        ci_upper=lambda df: np.mean(df.Income) + 1.96 * np.std(df.Income) / np.sqrt(len(df))
    )

result_final = atree_final.run(df)

# 2. Format statistics
format_statistics(result_final, format_spec="{mean:.0f} ({sd:.0f})", stat_name="Mean (SD)", inplace=True)
format_statistics(result_final, format_spec="[{ci_lower:.0f}, {ci_upper:.0f}]", stat_name="95% CI", inplace=True)

# 3. Create publication table
pub_table = simple_table(result_final, by="df.Country", pivot_statistics=True)
pub_table